In [ ]:
import pandas as pd
from itertools import combinations

df = pd.read_csv("/content/drive/MyDrive/drone iit/feature_matrix_t30.csv")

class_cols = [col for col in df.columns if "_class" in col]

for col in class_cols:
    df[col] = df[col].apply(lambda x: 1 if x != 0 else 0)

base_classes = list(range(1, 9))

combination_to_label = {}
next_label = 9

for r in range(2, len(base_classes) + 1):
    for combo in combinations(base_classes, r):
        combination_to_label[combo] = next_label
        next_label += 1

def map_classes(row):
    present = []

    for idx, col in enumerate(class_cols):
        if row[col] == 1:
            present.append(idx + 1)

    present = sorted(present)

    if len(present) == 0:
        return 0
    elif len(present) == 1:
        return present[0]
    else:
        return combination_to_label.get(tuple(present), 0)

df['Classes_Present'] = df.apply(map_classes, axis=1)

df.to_csv("/content/drive/MyDrive/drone iit/feature_matrix_final.csv", index=False)

print("✅ Done! Updated CSV saved as feature_matrix_final.csv")

✅ Done! Updated CSV saved as feature_matrix_final.csv


In [ ]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

CSV_PATH = "/content/drive/MyDrive/drone iit/feature_matrix_final.csv"

df = pd.read_csv(CSV_PATH)

print("Dataset shape:", df.shape)
print(df.head())

y = df['Classes_Present']

X = df.drop(columns=['Tile_Name', 'Classes_Present'])

print("\nFeature shape:", X.shape)
print("Target shape :", y.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)
print("\nTrain size:", X_train.shape)
print("Test size :", X_test.shape)

model = RandomForestClassifier(
    n_estimators=150,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

print("\n✅ Model training completed")

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"\n🎯 Accuracy: {accuracy*100:.2f}%")

print("\n📊 Classification Report:")
print(classification_report(y_test, y_pred))

print("\n📉 Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
cv_scores = cross_val_score(model, X, y, cv=5)

print("\n🔁 Cross Validation Scores:", cv_scores)
print(f"📈 Mean CV Accuracy: {cv_scores.mean()*100:.2f}%")

importances = model.feature_importances_
feature_names = X.columns

feat_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

print("\n🔥 Top 10 Important Features:")
print(feat_df.head(10))

import joblib

MODEL_PATH = "/content/drive/MyDrive/drone iit/model.pkl"
joblib.dump(model, MODEL_PATH)

print(f"\n💾 Model saved at: {MODEL_PATH}")

Dataset shape: (1340, 250)
  Tile_Name  Classes_Present  Utility_class  Water_Body_Pt_class  \
0   image_0               80              0                    0   
1   image_1               80              0                    0   
2   image_2               24              0                    0   
3   image_3               35              0                    0   
4   image_4                8              0                    0   

   RCC_Building_class  Utility_Poly_class  Tiled_class  Road_class  Tin_class  \
0                   1                   0            0           1          1   
1                   1                   0            0           1          1   
2                   1                   0            0           1          0   
3                   0                   0            0           1          0   
4                   0                   0            0           0          0   

   Water_Body_class  ...  Water_Body_Texture_homogeneity  \
0                

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/m


🔁 Cross Validation Scores: [0.90298507 0.90298507 0.89552239 0.89925373 0.88432836]
📈 Mean CV Accuracy: 89.70%

🔥 Top 10 Important Features:
                    Feature  Importance
158             Road_R_mean    0.018678
161              Road_R_max    0.018045
83    RCC_Building_Gray_max    0.017094
162             Road_G_mean    0.016301
186    Road_Saturation_mean    0.014574
187    Road_Brightness_mean    0.013816
80   RCC_Building_Gray_mean    0.013401
170          Road_Gray_mean    0.012926
91   RCC_Building_Perimeter    0.012920
180               Road_Area    0.012757

💾 Model saved at: /content/drive/MyDrive/drone iit/model.pkl
